# Novelty 2: confidence-threshold refusal gate (Qwen1.5-7B-Chat on a T4)

**What it does:** reads how likely Qwen is to start a refusal code (no text generation, no judge, no API key) and decides answer vs refuse with a threshold tuned on other source questions. Details and how to read the result: `README.md` in this folder.

**To run it:**
1. Runtime > Change runtime type > **T4 GPU**
2. *(Only if the GitHub repo is private)* key icon on the left > add a secret `GITHUB_TOKEN`, notebook access on
3. Runtime > **Run all**. Keep the tab open. If Colab disconnects, click Run all again: it resumes.

Results land in `MyDrive/refusalbench-results/novelty2_confidence_threshold/`. The key file is `RESULT_AUTO.md`. The last step downloads a zip of the small result files: put it in this folder.

In [ ]:
# NOVELTY 2 - confidence-threshold refusal gate (Qwen1.5-7B-Chat, T4). ONE cell: Runtime > Run all.
# Safe to re-run after a disconnect: finished work is kept on Google Drive and skipped.
# The first run downloads the model (about 15 GB, several minutes): progress is printed below.
import os, subprocess, sys

REPO = 'aidepartmentrah-pixel/refusalbench-reproduction'   # the repo that holds this code (change if you push elsewhere)
OUT  = '/content/drive/MyDrive/refusalbench-results'        # results are saved here, on your Drive


def run(cmd, **kw):
    """Run a command and print its output LIVE (a plain subprocess.run shows nothing in a notebook)."""
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1'}, **kw)
    while True:
        chunk = os.read(p.stdout.fileno(), 4096)
        if not chunk:
            break
        print(chunk.decode('utf-8', 'replace'), end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'command failed (exit {p.returncode}): {cmd}')


from google.colab import drive
drive.mount('/content/drive')

token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''              # only needed if the repo is private
except Exception:
    pass
url = f'https://{token + "@" if token else ""}github.com/{REPO}.git'
if not os.path.exists('/content/refusalbench-reproduction'):
    run(['git', 'clone', '-q', url, '/content/refusalbench-reproduction'])
os.chdir('/content/refusalbench-reproduction')
run(['git', 'pull', '-q'])
print('installing packages ...')
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])

# GPU check
run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'])

# The experiment: scores all 1,600 examples (about 30-60 min on a T4, an estimate), then analyses. No judge, no API key.
BASELINE = f'{OUT}/qwen15_7b_baseline'                       # optional: adds a reference line if the baseline run is there
cmd = [sys.executable, 'Benchmark-Novelty-2-Confidence-Threshold/code/run_novelty2.py', '--out', OUT]
if os.path.exists(f'{BASELINE}/target_outputs.jsonl'):
    cmd += ['--baseline-dir', BASELINE]
run(cmd)

# Show the figures and make a zip to download (the Drive copy is already saved)
from IPython.display import Image, display
R = f'{OUT}/novelty2_confidence_threshold'
for f in ('tradeoff.png', 's1_distribution.png'):
    if os.path.exists(f'{R}/{f}'):
        display(Image(f'{R}/{f}'))
try:
    from google.colab import files
    run(f'cd {OUT} && zip -qr /content/novelty2_results.zip novelty2_confidence_threshold -x "*/shards/*"', shell=True)
    files.download('/content/novelty2_results.zip')
except Exception as e:
    print('download skipped:', e)
